In [1]:
import os

BASE_DIR = "/kaggle/working/spectral_v2"
DIRS = {
    "generations": f"{BASE_DIR}/generations", "extractions": f"{BASE_DIR}/extractions",
    "features": f"{BASE_DIR}/features", "results": f"{BASE_DIR}/results", "logs": f"{BASE_DIR}/logs",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print(os.listdir("/kaggle/input/notebooks/aishidev"))

['final-spectral-research-project-gsm8k-features', 'final-spectral-research-project-arc-features']


In [2]:
import shutil

os.makedirs(f"{DIRS['features']}", exist_ok=True)

GSM8K_SOURCE = "/kaggle/input/notebooks/aishidev/final-spectral-research-project-gsm8k-features/spectral_v2/features"
ARC_SOURCE = "/kaggle/input/notebooks/aishidev/final-spectral-research-project-arc-features/spectral_v2/features"

shutil.copytree(GSM8K_SOURCE, DIRS["features"], dirs_exist_ok=True)
shutil.copytree(ARC_SOURCE, DIRS["features"], dirs_exist_ok=True)

print("Files in features dir:", os.listdir(DIRS["features"]))

Files in features dir: ['metrics_per_example_layer_arc.csv', 'metrics_per_example_layer_gsm8k_NORMALIZED.csv', 'step9_master_table.csv', 'metrics_per_example_layer_arc_NORMALIZED.csv', 'metrics_per_example_layer_gsm8k.csv', 'arc_step9_master_table.csv']


In [3]:
# ============================================================
# RESEARCH STEP 8 — Formal statistical testing with BH correction
# Runs for both GSM8K and ARC-Challenge feature tables.
# Uses Mann-Whitney U (matches the original pilot's method),
# then applies Benjamini-Hochberg FDR correction across all
# 96 layer x metric comparisons per dataset.
# No GPU needed.
# ============================================================

In [4]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

METRICS = ["fiedler", "spectral_entropy", "hfer", "smoothness"]

def run_bh_correction(features_path, dataset_name):
    df = pd.read_csv(features_path)
    print(f"\n{'='*60}")
    print(f"{dataset_name}: BH correction across all layer x metric tests")
    print(f"{'='*60}")

    rows = []
    for metric in METRICS:
        for layer in sorted(df["layer"].unique()):
            layer_df = df[df["layer"] == layer]
            correct_vals = layer_df[layer_df["is_correct"] == True][metric]
            incorrect_vals = layer_df[layer_df["is_correct"] == False][metric]

            # Mann-Whitney U -- matches the original pilot's method, non-parametric
            stat, p_raw = stats.mannwhitneyu(correct_vals, incorrect_vals, alternative="two-sided")

            n1, n2 = len(correct_vals), len(incorrect_vals)
            pooled_std = np.sqrt(((n1-1)*correct_vals.std(ddof=1)**2 + (n2-1)*incorrect_vals.std(ddof=1)**2) / (n1+n2-2))
            d = 0.0 if pooled_std == 0 else (correct_vals.mean() - incorrect_vals.mean()) / pooled_std

            rows.append({"metric": metric, "layer": layer, "cohens_d": d, "p_raw": p_raw})

    results_df = pd.DataFrame(rows)

    # Apply BH correction across all 96 tests
    reject, p_adjusted, _, _ = multipletests(results_df["p_raw"].values, alpha=0.05, method="fdr_bh")
    results_df["p_adjusted"] = p_adjusted
    results_df["significant_after_bh"] = reject

    n_raw_sig = (results_df["p_raw"] < 0.05).sum()
    n_bh_sig = results_df["significant_after_bh"].sum()

    print(f"Total tests: {len(results_df)}")
    print(f"Significant at raw p < 0.05 (uncorrected): {n_raw_sig}")
    print(f"Significant after BH correction: {n_bh_sig}")

    if n_bh_sig > 0:
        print(f"\nSurvivors after BH correction:")
        survivors = results_df[results_df["significant_after_bh"]].sort_values("p_adjusted")
        print(survivors[["metric", "layer", "cohens_d", "p_raw", "p_adjusted"]].to_string(index=False))
    else:
        print("\n>>> NO layer x metric combinations survive BH correction. <<<")
        print(">>> This means none of the raw effects are statistically distinguishable from noise <<<")
        print(">>> once properly corrected for the 96 comparisons tested. <<<")

    return results_df

# Run for both datasets
gsm8k_results = run_bh_correction(f"{DIRS['features']}/metrics_per_example_layer_gsm8k.csv", "GSM8K")
gsm8k_results.to_csv(f"{DIRS['results']}/gsm8k_bh_corrected_results.csv", index=False)

arc_results = run_bh_correction(f"{DIRS['features']}/metrics_per_example_layer_arc.csv", "ARC-CHALLENGE")
arc_results.to_csv(f"{DIRS['results']}/arc_bh_corrected_results.csv", index=False)

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"GSM8K: {gsm8k_results['significant_after_bh'].sum()} / 96 survive BH correction")
print(f"ARC:   {arc_results['significant_after_bh'].sum()} / 96 survive BH correction")


GSM8K: BH correction across all layer x metric tests
Total tests: 96
Significant at raw p < 0.05 (uncorrected): 65
Significant after BH correction: 64

Survivors after BH correction:
          metric  layer  cohens_d        p_raw  p_adjusted
         fiedler     12  0.831432 4.043433e-07    0.000035
spectral_entropy     14 -0.879651 7.303147e-07    0.000035
         fiedler     14  0.658859 1.432828e-06    0.000046
spectral_entropy     12 -0.799837 2.169628e-06    0.000052
spectral_entropy     20 -0.776976 7.907536e-06    0.000133
spectral_entropy      6 -0.778371 8.338850e-06    0.000133
         fiedler      6  0.780290 1.066667e-05    0.000146
spectral_entropy     10 -0.746168 2.779566e-05    0.000267
      smoothness     23  0.771878 2.599493e-05    0.000267
            hfer      8 -0.674418 2.272087e-05    0.000267
         fiedler     10  0.734735 4.947944e-05    0.000432
         fiedler     15  0.634727 1.584610e-04    0.000922
            hfer     20  0.645627 1.490836e-04   

In [5]:
import shutil

SOURCE = "/kaggle/input/notebooks/aishidev/final-spectral-research-project-gsm8k-features/spectral_v2"
shutil.copytree(SOURCE, BASE_DIR, dirs_exist_ok=True)

print(os.path.exists(f"{DIRS['features']}/metrics_per_example_layer_gsm8k_NORMALIZED.csv"))

True


In [6]:
# ============================================================
# RESEARCH STEP 8 (NORMALIZED LAPLACIAN VERSION)
# Same BH correction procedure as the combinatorial version,
# applied to the normalized-Laplacian feature tables for both
# datasets. Closes the gap: checks whether GSM8K's layer-16
# survivor (and any ARC normalized candidates) hold up under
# formal multiple-comparison correction.
# No GPU needed.
# ============================================================

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

METRICS = ["fiedler", "spectral_entropy", "hfer", "smoothness"]

def run_bh_correction(features_path, dataset_name):
    df = pd.read_csv(features_path)
    print(f"\n{'='*60}")
    print(f"{dataset_name} (NORMALIZED Laplacian): BH correction across all layer x metric tests")
    print(f"{'='*60}")

    rows = []
    for metric in METRICS:
        for layer in sorted(df["layer"].unique()):
            layer_df = df[df["layer"] == layer]
            correct_vals = layer_df[layer_df["is_correct"] == True][metric]
            incorrect_vals = layer_df[layer_df["is_correct"] == False][metric]

            stat, p_raw = stats.mannwhitneyu(correct_vals, incorrect_vals, alternative="two-sided")

            n1, n2 = len(correct_vals), len(incorrect_vals)
            pooled_std = np.sqrt(((n1-1)*correct_vals.std(ddof=1)**2 + (n2-1)*incorrect_vals.std(ddof=1)**2) / (n1+n2-2))
            d = 0.0 if pooled_std == 0 else (correct_vals.mean() - incorrect_vals.mean()) / pooled_std

            rows.append({"metric": metric, "layer": layer, "cohens_d": d, "p_raw": p_raw})

    results_df = pd.DataFrame(rows)
    reject, p_adjusted, _, _ = multipletests(results_df["p_raw"].values, alpha=0.05, method="fdr_bh")
    results_df["p_adjusted"] = p_adjusted
    results_df["significant_after_bh"] = reject

    n_raw_sig = (results_df["p_raw"] < 0.05).sum()
    n_bh_sig = results_df["significant_after_bh"].sum()

    print(f"Total tests: {len(results_df)}")
    print(f"Significant at raw p < 0.05 (uncorrected): {n_raw_sig}")
    print(f"Significant after BH correction: {n_bh_sig}")

    if n_bh_sig > 0:
        print(f"\nSurvivors after BH correction:")
        survivors = results_df[results_df["significant_after_bh"]].sort_values("p_adjusted")
        print(survivors[["metric", "layer", "cohens_d", "p_raw", "p_adjusted"]].to_string(index=False))
    else:
        print("\n>>> NO layer x metric combinations survive BH correction. <<<")

    # Specifically flag the candidate of interest
    return results_df

gsm8k_norm_results = run_bh_correction(
    f"{DIRS['features']}/metrics_per_example_layer_gsm8k_NORMALIZED.csv", "GSM8K"
)
gsm8k_norm_results.to_csv(f"{DIRS['results']}/gsm8k_NORMALIZED_bh_corrected_results.csv", index=False)

print(f"\n--- Layer 16, spectral_entropy specifically ---")
layer16_row = gsm8k_norm_results[(gsm8k_norm_results["metric"] == "spectral_entropy") & (gsm8k_norm_results["layer"] == 16)]
print(layer16_row.to_string(index=False))

# Also run for ARC's normalized data, for full consistency
try:
    arc_norm_results = run_bh_correction(
        f"{DIRS['features']}/metrics_per_example_layer_arc_NORMALIZED.csv", "ARC-CHALLENGE"
    )
    arc_norm_results.to_csv(f"{DIRS['results']}/arc_NORMALIZED_bh_corrected_results.csv", index=False)
except FileNotFoundError:
    print("\n(ARC normalized feature table not found in this session -- skipping ARC check.)")

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"GSM8K (normalized): {gsm8k_norm_results['significant_after_bh'].sum()} / 96 survive BH correction")


GSM8K (NORMALIZED Laplacian): BH correction across all layer x metric tests
Total tests: 96
Significant at raw p < 0.05 (uncorrected): 69
Significant after BH correction: 68

Survivors after BH correction:
          metric  layer  cohens_d        p_raw  p_adjusted
         fiedler     12  0.827609 5.773760e-07    0.000053
         fiedler     14  0.655914 1.096497e-06    0.000053
spectral_entropy     16 -0.813745 2.566016e-06    0.000082
spectral_entropy     11 -0.741577 1.535357e-05    0.000368
         fiedler     10  0.741315 2.430551e-05    0.000467
spectral_entropy     21 -0.727592 3.123573e-05    0.000500
spectral_entropy     22 -0.754432 3.685838e-05    0.000505
         fiedler     19  0.645953 7.276768e-05    0.000635
spectral_entropy     17 -0.754679 7.161884e-05    0.000635
         fiedler     15  0.653182 5.910549e-05    0.000635
spectral_entropy     20 -0.703011 6.937243e-05    0.000635
spectral_entropy     13 -0.703325 1.045335e-04    0.000836
spectral_entropy      9 -0